In [1]:
# 1. Imports
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

In [2]:

# 2. Configurable hyperparameters
config = {
    "n_splits": 5,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 6,
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "scale_pos_weight": 6.4,
    "experiment_name": "loan-risk-xgboost",
    "run_name": "xgb-kfold-binary",
    "save_path": "loan_risk_model.pth",
    "dataset_path": "/mnt/object/train_transformed_1.csv",
    "label_col": "risk_level"
}

In [3]:
# 3. Load dataset
df = pd.read_csv(config["dataset_path"])
X = df.drop(columns=[config["label_col"]])
y = df[config["label_col"]]  # Already binary-mapped in transform step


In [4]:
# 4. Set MLflow experiment
mlflow.set_experiment(config["experiment_name"])

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1746921146045, experiment_id='1', last_update_time=1746921146045, lifecycle_stage='active', name='loan-risk-xgboost', tags={}>

In [5]:
# 5. Start run
with mlflow.start_run(run_name=config["run_name"]):
    mlflow.log_params({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "n_splits": config["n_splits"],
        "scale_pos_weight": config["scale_pos_weight"]
    })

    kf = StratifiedKFold(n_splits=config["n_splits"], shuffle=True, random_state=config["random_state"])
    fold_accuracies = []
    fold_confidences = {0: [], 1: []}
    all_f1_macro = []
    all_f1_weighted = []
    all_preds = []
    all_probs = []
    all_true = []

    for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y.iloc[train_index], y.iloc[val_index]

        model = XGBClassifier(
            n_estimators=config["n_estimators"],
            learning_rate=config["learning_rate"],
            max_depth=config["max_depth"],
            random_state=config["random_state"],
            objective=config["objective"],
            eval_metric=config["eval_metric"],
            scale_pos_weight=config["scale_pos_weight"]
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)[:, 1]

        # Save for global evaluation
        all_preds.extend(y_pred)
        all_probs.extend(y_prob)
        all_true.extend(y_val)

        # Track confidence per predicted class
        for pred_label, prob in zip(y_pred, y_prob):
            fold_confidences[pred_label].append(prob)

        acc = accuracy_score(y_val, y_pred)
        fold_accuracies.append(acc)

        print(f"Fold {fold+1} Accuracy: {acc:.4f}")
        print(classification_report(y_val, y_pred, target_names=["Low (0)", "High (1)"]))

        mlflow.log_metric(f"fold_{fold+1}_accuracy", acc)

        # F1, precision, recall, support per class
        prec, rec, f1, support = precision_recall_fscore_support(y_val, y_pred, labels=[0, 1], zero_division=0)
        for cls_idx, cls_name in zip([0, 1], ["Low", "High"]):
            mlflow.log_metric(f"fold_{fold+1}_precision_{cls_name}", prec[cls_idx])
            mlflow.log_metric(f"fold_{fold+1}_recall_{cls_name}", rec[cls_idx])
            mlflow.log_metric(f"fold_{fold+1}_f1_{cls_name}", f1[cls_idx])
            mlflow.log_metric(f"fold_{fold+1}_support_{cls_name}", support[cls_idx])

        f1_macro = f1_score(y_val, y_pred, average="macro", zero_division=0)
        f1_weighted = f1_score(y_val, y_pred, average="weighted", zero_division=0)
        all_f1_macro.append(f1_macro)
        all_f1_weighted.append(f1_weighted)

        mlflow.log_metric(f"fold_{fold+1}_f1_macro", f1_macro)
        mlflow.log_metric(f"fold_{fold+1}_f1_weighted", f1_weighted)

    # 6. Log overall averages
    avg_acc = np.mean(fold_accuracies)
    mlflow.log_metric("avg_kfold_accuracy", avg_acc)

    for class_label in [0, 1]:
        if fold_confidences[class_label]:
            avg_conf = np.mean(fold_confidences[class_label])
            mlflow.log_metric(f"avg_confidence_class_{class_label}", avg_conf)
            print(f"Avg confidence for class {class_label}: {avg_conf:.4f}")

    mlflow.log_metric("avg_f1_macro", np.mean(all_f1_macro))
    mlflow.log_metric("avg_f1_weighted", np.mean(all_f1_weighted))

    # 7. Log overall ROC-AUC
    roc_auc = roc_auc_score(all_true, all_probs)
    mlflow.log_metric("roc_auc_overall", roc_auc)
    print(f"ROC-AUC Overall: {roc_auc:.4f}")

    # 8. Log overall confusion matrix
    cm = confusion_matrix(all_true, all_preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low", "High"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title("Confusion Matrix (Overall)")
    cm_path = "confusion_matrix_overall.png"
    plt.savefig(cm_path)
    plt.close()
    mlflow.log_artifact(cm_path)

    # 9. Save model
    joblib.dump(model, config["save_path"])
    mlflow.log_artifact(config["save_path"])

    print(f"✅ Model saved to {config['save_path']}")
    print(f"📊 KFold avg accuracy: {avg_acc:.4f}")

Fold 1 Accuracy: 0.8661
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312811
      Medium       0.00      0.00      0.00      2045
        High       0.66      0.02      0.04     46851

    accuracy                           0.87    361707
   macro avg       0.51      0.34      0.32    361707
weighted avg       0.83      0.87      0.81    361707



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 2 Accuracy: 0.8661
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312811
      Medium       0.00      0.00      0.00      2046
        High       0.66      0.02      0.04     46850

    accuracy                           0.87    361707
   macro avg       0.51      0.34      0.32    361707
weighted avg       0.83      0.87      0.81    361707



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 3 Accuracy: 0.8662
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312811
      Medium       0.00      0.00      0.00      2046
        High       0.67      0.02      0.04     46850

    accuracy                           0.87    361707
   macro avg       0.51      0.34      0.32    361707
weighted avg       0.84      0.87      0.81    361707



...

Fold 4 Accuracy: 0.8662
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312811
      Medium       0.00      0.00      0.00      2046
        High       0.67      0.02      0.04     46850

    accuracy                           0.87    361707
   macro avg       0.51      0.34      0.32    361707
weighted avg       0.84      0.87      0.81    361707



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 5 Accuracy: 0.8660
              precision    recall  f1-score   support

         Low       0.87      1.00      0.93    312810
      Medium       0.00      0.00      0.00      2045
        High       0.65      0.02      0.04     46851

    accuracy                           0.87    361706
   macro avg       0.51      0.34      0.32    361706
weighted avg       0.83      0.87      0.81    361706



/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


✅ Model saved to loan_risk_model.pth
📊 KFold avg accuracy: 0.8661
🏃 View run xgb-kfold-run at: http://129.114.25.120:8000/#/experiments/1/runs/3e71ddddda1e4b0188eb695306acc0d9
🧪 View experiment at: http://129.114.25.120:8000/#/experiments/1
